In [5]:
import warnings
warnings.filterwarnings("ignore")

import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

from faker import Faker

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from pathlib import Path

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
Faker.seed(SEED)

fake = Faker()

In [7]:
ROOT = Path.cwd()

DATA_DIR = ROOT / "data"
SYNTHETIC_DIR = DATA_DIR / "synthetic"

FIGURE_DIR = ROOT / "figures"

OUTPUT_DIR = ROOT / "outputs"

for directory in [
    DATA_DIR,
    SYNTHETIC_DIR,
    FIGURE_DIR,
    OUTPUT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [8]:
CONFIG = {

    "records":10000,

    "seed":42,

    "dataset_name":"SAFE_MFI_Synthetic_Dataset_v1",

    "country":"Nigeria",

    "currency":"NGN",

    "train_split":0.80,

    "test_split":0.20

}

CONFIG

{'records': 10000,
 'seed': 42,
 'dataset_name': 'SAFE_MFI_Synthetic_Dataset_v1',
 'country': 'Nigeria',
 'currency': 'NGN',
 'train_split': 0.8,
 'test_split': 0.2}

In [9]:
REGIONS = [

    "North",

    "South",

    "East",

    "West",

    "Central"

]

GENDER = [

    "Male",

    "Female"

]

MARITAL_STATUS = [

    "Single",

    "Married",

    "Divorced",

    "Widowed"

]

EDUCATION = [

    "None",

    "Primary",

    "Secondary",

    "Tertiary"

]

EMPLOYMENT = [

    "Salaried",

    "Self-employed",

    "Farmer",

    "Trader",

    "Artisan"

]

BUSINESS_TYPE = [

    "Retail",

    "Agriculture",

    "Manufacturing",

    "Transport",

    "Services"

]

REPAYMENT = [

    "Good",

    "Average",

    "Poor"

]

RURAL_URBAN = [

    "Urban",

    "Rural"

]

In [10]:
def generate_income():

    income = np.random.lognormal(
        mean=np.log(90000),
        sigma=0.45
    )

    return round(max(income,20000),2)


def generate_savings(income):

    savings = np.random.gamma(
        shape=2.5,
        scale=income/8
    )

    return round(savings,2)


def generate_expenses(income):

    expenses = income * np.random.uniform(
        0.35,
        0.80
    )

    return round(expenses,2)


def generate_debt(income):

    debt = np.random.gamma(
        shape=2,
        scale=income/15
    )

    return round(debt,2)


def generate_age():

    return int(
        np.clip(
            np.random.normal(
                loc=38,
                scale=10
            ),
            18,
            70
        )
    )


def generate_household():

    return max(
        1,
        np.random.poisson(4)
    )

In [11]:
def generate_loan_amount(income):

    amount = np.random.gamma(
        shape=2.5,
        scale=income/2.5
    )

    amount = min(
        amount,
        income*8
    )

    return round(amount,2)


def generate_loan_term():

    return random.choice(

        [6,12,18,24,36]

    )


def generate_interest():

    return round(

        np.random.uniform(8,28),

        2

    )

In [12]:
def random_date():

    start = datetime(2023,1,1)

    end = datetime(2025,12,31)

    delta = end - start

    return (

        start +

        timedelta(

            days=random.randint(

                0,

                delta.days

            )

        )

    )


def generate_model_version():

    return "SAFE-MFI-v1"


def generate_branch():

    return f"BR{random.randint(1,50):03d}"


def customer_id(i):

    return f"CUST{i:06d}"

In [13]:
print("Income:", generate_income())

print("Savings:", generate_savings(100000))

print("Expenses:", generate_expenses(100000))

print("Debt:", generate_debt(100000))

print("Age:", generate_age())

print("Loan:", generate_loan_amount(120000))

print("Interest:", generate_interest())

print("Branch:", generate_branch())

print("Customer:", customer_id(1))

Income: 112542.51
Savings: 24618.16
Expenses: 61939.63
Debt: 9215.22
Age: 35
Loan: 124987.47
Interest: 8.41
Branch: BR041
Customer: CUST000001


In [14]:
def debt_to_income_ratio(debt, income):
    return round(debt / max(income, 1), 3)


def savings_to_income_ratio(savings, income):
    return round(savings / max(income, 1), 3)


def income_stability():
    return random.choice(
        [
            "Stable",
            "Moderately Stable",
            "Unstable"
        ]
    )


def repayment_history():
    return random.choices(
        population=[
            "Good",
            "Average",
            "Poor"
        ],
        weights=[0.55, 0.30, 0.15]
    )[0]


def previous_defaults():

    return random.choices(
        [0,1,2],
        weights=[0.75,0.20,0.05]
    )[0]


def mobile_money_score():

    score=np.random.beta(
        5,
        2
    )*100

    return round(score,1)


def credit_history():

    return random.randint(
        0,
        20
    )


def business_age():

    return random.randint(
        0,
        20
    )


def collateral():

    return random.choices(
        ["Yes","No"],
        weights=[0.45,0.55]
    )[0]


def group_lending():

    return random.choices(
        ["Yes","No"],
        weights=[0.65,0.35]
    )[0]

In [15]:
def calculate_risk_score(record):

    score = 0

    # Income

    if record["monthly_income"] > 150000:
        score += 2

    elif record["monthly_income"] > 100000:
        score += 1

    # Savings

    if record["savings_balance"] > 80000:
        score += 2

    # Repayment behaviour

    if record["repayment_history"] == "Good":
        score += 2

    elif record["repayment_history"] == "Average":
        score += 1

    else:
        score -= 3

    # Previous defaults

    score -= record["previous_defaults"] * 2

    # Debt burden

    if record["debt_to_income_ratio"] > 0.60:
        score -= 2

    elif record["debt_to_income_ratio"] > 0.40:
        score -= 1

    # Business maturity

    if record["business_age_years"] >= 5:
        score += 1

    # Credit history

    if record["credit_history_years"] >= 5:
        score += 1

    # Group lending

    if record["group_lending"] == "Yes":
        score += 1

    # Collateral

    if record["collateral"] == "Yes":
        score += 1

    # Mobile money usage

    if record["mobile_money_score"] >= 80:
        score += 1

    return score

In [16]:
def probability_of_repayment(score):

    probability = 1 / (1 + np.exp(-score / 2))

    return round(probability,4)

In [17]:
def generate_borrower(i):

    income = generate_income()

    savings = generate_savings(income)

    expenses = generate_expenses(income)

    debt = generate_debt(income)

    loan_amount = generate_loan_amount(income)

    borrower = {

        "customer_id": customer_id(i),

        "branch_id": generate_branch(),

        "application_date": random_date(),

        "region": random.choice(REGIONS),

        "rural_urban": random.choice(RURAL_URBAN),

        "age": generate_age(),

        "gender": random.choice(GENDER),

        "marital_status": random.choice(MARITAL_STATUS),

        "household_size": generate_household(),

        "education_level": random.choice(EDUCATION),

        "employment_status": random.choice(EMPLOYMENT),

        "business_type": random.choice(BUSINESS_TYPE),

        "business_age_years": business_age(),

        "monthly_income": income,

        "monthly_expenses": expenses,

        "savings_balance": savings,

        "existing_debt": debt,

        "debt_to_income_ratio":
            debt_to_income_ratio(
                debt,
                income
            ),

        "savings_to_income_ratio":
            savings_to_income_ratio(
                savings,
                income
            ),

        "income_stability":
            income_stability(),

        "collateral":
            collateral(),

        "credit_history_years":
            credit_history(),

        "previous_defaults":
            previous_defaults(),

        "mobile_money_score":
            mobile_money_score(),

        "loan_amount":
            loan_amount,

        "loan_term_months":
            generate_loan_term(),

        "interest_rate":
            generate_interest(),

        "repayment_history":
            repayment_history(),

        "loan_cycle":
            random.randint(1,6),

        "group_lending":
            group_lending(),

        "model_version":
            generate_model_version(),

        "decision_timestamp":
            random_date(),

        "monitoring_date":
            random_date()

    }

    score = calculate_risk_score(
        borrower
    )

    borrower["risk_score"] = score

    borrower["prediction_probability"] = \
        probability_of_repayment(score)

    borrower["risk_category"] = \
        "Low" if score >= 5 \
        else "Medium" if score >= 2 \
        else "High"

    borrower["human_review"] = \
        "Yes" if score < 2 else "No"

    borrower["manual_override"] = \
        random.choices(
            ["Yes","No"],
            weights=[0.05,0.95]
        )[0]

    borrower["drift_score"] = round(
        np.random.beta(2,8),
        3
    )

    borrower["loan_status"] = np.random.binomial(
        1,
        borrower["prediction_probability"]
    )

    return borrower

In [18]:
records = []

for i in range(
    1,
    CONFIG["records"] + 1
):

    records.append(

        generate_borrower(i)

    )

dataset = pd.DataFrame(records)

print(
    dataset.shape
)

dataset.head()

(10000, 40)


,customer_id,branch_id,application_date,region,rural_urban,age,gender,marital_status,household_size,education_level,...,model_version,decision_timestamp,monitoring_date,risk_score,prediction_probability,risk_category,human_review,manual_override,drift_score,loan_status
0,CUST000001,BR008,2023-02-21,East,Urban,32,Male,Married,2,None,...,SAFE-MFI-v1,2025-05-09,2024-03-27,6,0.9526,Low,No,No,0.224,1
1,CUST000002,BR018,2023-01-14,South,Rural,46,Female,Divorced,3,Primary,...,SAFE-MFI-v1,2023-09-13,2025-02-14,2,0.7311,Medium,No,No,0.296,1
2,CUST000003,BR019,2025-01-10,Central,Urban,46,Male,Single,3,Primary,...,SAFE-MFI-v1,2024-06-30,2023-05-27,6,0.9526,Low,No,No,0.405,1
3,CUST000004,BR011,2025-12-29,South,Urban,38,Female,Widowed,4,Secondary,...,SAFE-MFI-v1,2024-10-06,2024-03-11,4,0.8808,Medium,No,No,0.116,0
4,CUST000005,BR026,2025-07-28,South,Rural,23,Male,Married,1,Secondary,...,SAFE-MFI-v1,2023-11-24,2025-05-14,5,0.9241,Low,No,No,0.142,1


In [19]:
print("="*60)

print("Dataset Shape")

print(dataset.shape)

print("="*60)

print("Missing Values")

print(dataset.isnull().sum())

print("="*60)

print("Duplicate Records")

print(dataset.duplicated().sum())

print("="*60)

print(dataset.head())

Dataset Shape
(10000, 40)
Missing Values
customer_id                0
branch_id                  0
application_date           0
region                     0
rural_urban                0
age                        0
gender                     0
marital_status             0
household_size             0
education_level            0
employment_status          0
business_type              0
business_age_years         0
monthly_income             0
monthly_expenses           0
savings_balance            0
existing_debt              0
debt_to_income_ratio       0
savings_to_income_ratio    0
income_stability           0
collateral                 0
credit_history_years       0
previous_defaults          0
mobile_money_score         0
loan_amount                0
loan_term_months           0
interest_rate              0
repayment_history          0
loan_cycle                 0
group_lending              0
model_version              0
decision_timestamp         0
monitoring_date            0
ri

In [20]:
print("=" * 70)
print("SAFE-MFI DATASET VALIDATION")
print("=" * 70)

print(f"Dataset Shape: {dataset.shape}")

print("\nMissing Values")
print("-" * 70)
print(dataset.isnull().sum())

print("\nDuplicate Records")
print("-" * 70)
print(dataset.duplicated().sum())

print("\nData Types")
print("-" * 70)
print(dataset.dtypes)

print("\nLoan Status Distribution")
print("-" * 70)
print(dataset["loan_status"].value_counts())
print()
print(dataset["loan_status"].value_counts(normalize=True) * 100)

print("\nRisk Category Distribution")
print("-" * 70)
print(dataset["risk_category"].value_counts())

print("\nHuman Review Rate")
print("-" * 70)
print(dataset["human_review"].value_counts(normalize=True) * 100)

print("\nManual Override Rate")
print("-" * 70)
print(dataset["manual_override"].value_counts(normalize=True) * 100)

SAFE-MFI DATASET VALIDATION
Dataset Shape: (10000, 40)

Missing Values
----------------------------------------------------------------------
customer_id                0
branch_id                  0
application_date           0
region                     0
rural_urban                0
age                        0
gender                     0
marital_status             0
household_size             0
education_level            0
employment_status          0
business_type              0
business_age_years         0
monthly_income             0
monthly_expenses           0
savings_balance            0
existing_debt              0
debt_to_income_ratio       0
savings_to_income_ratio    0
income_stability           0
collateral                 0
credit_history_years       0
previous_defaults          0
mobile_money_score         0
loan_amount                0
loan_term_months           0
interest_rate              0
repayment_history          0
loan_cycle                 0
group_lending    

In [21]:
summary = dataset.describe(include="all")

summary

,customer_id,branch_id,application_date,region,rural_urban,age,gender,marital_status,household_size,education_level,...,model_version,decision_timestamp,monitoring_date,risk_score,prediction_probability,risk_category,human_review,manual_override,drift_score,loan_status
count,10000,10000,10000,10000,10000,10000.000000,10000,10000,10000.000000,10000,...,10000,10000,10000,10000.000000,10000.000000,10000,10000,10000,10000.000000,10000.000000
unique,10000,50,NaN,5,2,NaN,2,4,NaN,4,...,1,NaN,NaN,NaN,NaN,3,2,2,NaN,NaN
top,CUST009984,BR050,NaN,South,Urban,NaN,Female,Divorced,NaN,None,...,SAFE-MFI-v1,NaN,NaN,NaN,NaN,Low,No,No,NaN,NaN
freq,1,236,NaN,2044,5064,NaN,5087,2537,NaN,2545,...,10000,NaN,NaN,NaN,NaN,4670,8315,9506,NaN,NaN
mean,NaN,NaN,2024-06-30 15:30:31.680000,NaN,NaN,37.557400,NaN,NaN,4.013600,NaN,...,NaN,2024-07-02 13:19:12,2024-07-02 16:09:59.040000,3.948600,0.825104,NaN,NaN,NaN,0.198827,0.824300
min,NaN,NaN,2023-01-01 00:00:00,NaN,NaN,18.000000,NaN,NaN,1.000000,NaN,...,NaN,2023-01-01 00:00:00,2023-01-01 00:00:00,-7.000000,0.029300,NaN,NaN,NaN,0.001000,0.000000
25%,NaN,NaN,2023-10-04 00:00:00,NaN,NaN,31.000000,NaN,NaN,3.000000,NaN,...,NaN,2023-09-28 00:00:00,2023-09-29 00:00:00,3.000000,0.817600,NaN,NaN,NaN,0.107000,1.000000
50%,NaN,NaN,2024-07-03 12:00:00,NaN,NaN,37.000000,NaN,NaN,4.000000,NaN,...,NaN,2024-07-02 00:00:00,2024-07-06 00:00:00,4.000000,0.880800,NaN,NaN,NaN,0.178000,1.000000
75%,NaN,NaN,2025-03-30 00:00:00,NaN,NaN,44.000000,NaN,NaN,5.000000,NaN,...,NaN,2025-04-08 00:00:00,2025-04-03 00:00:00,6.000000,0.952600,NaN,NaN,NaN,0.270000,1.000000
max,NaN,NaN,2025-12-31 00:00:00,NaN,NaN,70.000000,NaN,NaN,13.000000,NaN,...,NaN,2025-12-31 00:00:00,2025-12-31 00:00:00,11.000000,0.995900,NaN,NaN,NaN,0.759000,1.000000


In [22]:
numeric_columns = dataset.select_dtypes(
    include=[
        np.number
    ]
)

correlation = numeric_columns.corr()

correlation

,age,household_size,business_age_years,monthly_income,monthly_expenses,savings_balance,existing_debt,debt_to_income_ratio,savings_to_income_ratio,credit_history_years,previous_defaults,mobile_money_score,loan_amount,loan_term_months,interest_rate,loan_cycle,risk_score,prediction_probability,drift_score,loan_status
age,1.000000,0.000790,-0.001966,0.000712,0.002769,0.007739,0.000782,-0.005559,0.003012,0.002965,-0.004707,0.005626,0.003544,0.010064,0.015507,0.009496,0.016596,0.020025,0.015090,0.021747
household_size,0.000790,1.000000,0.003965,0.003236,0.001372,0.000388,0.003861,-0.000613,-0.001671,-0.006463,0.004577,0.003913,-0.000466,-0.021779,0.004840,0.019204,-0.009678,-0.010569,-0.003601,-0.005654
business_age_years,-0.001966,0.003965,1.000000,0.001206,0.007507,0.005675,-0.005390,-0.014701,0.009957,0.019765,0.007284,0.000077,-0.004168,0.011875,-0.001165,0.003310,0.116022,0.090268,0.012287,0.030648
monthly_income,0.000712,0.003236,0.001206,1.000000,0.888690,0.565220,0.517546,-0.009189,0.000574,0.001448,-0.010587,-0.019363,0.583380,0.006580,-0.017453,0.005032,0.334905,0.225923,-0.005312,0.098677
monthly_expenses,0.002769,0.001372,0.007507,0.888690,1.000000,0.500152,0.457474,-0.007661,-0.000808,0.000850,-0.006809,-0.022844,0.516877,0.007140,-0.014357,0.002088,0.291297,0.194800,-0.003807,0.092690
savings_balance,0.007739,0.000388,0.005675,0.565220,0.500152,1.000000,0.295915,-0.003504,0.743882,0.002491,-0.005081,-0.005979,0.311078,-0.004103,-0.021176,0.004100,0.280505,0.175936,-0.008325,0.079217
existing_debt,0.000782,0.003861,-0.005390,0.517546,0.457474,0.295915,1.000000,0.771681,0.002076,-0.013356,-0.000955,-0.001944,0.307186,-0.008914,-0.018898,0.016663,0.149817,0.093635,-0.000985,0.027598
debt_to_income_ratio,-0.005559,-0.000613,-0.014701,-0.009189,-0.007661,-0.003504,0.771681,1.000000,0.004997,-0.005268,0.003609,0.015529,-0.001832,-0.006587,-0.005774,0.012704,-0.034987,-0.035932,0.001505,-0.035359
savings_to_income_ratio,0.003012,-0.001671,0.009957,0.000574,-0.000808,0.743882,0.002076,0.004997,1.000000,0.005044,0.000048,0.005749,-0.017526,-0.009578,-0.016532,-0.005626,0.093189,0.055537,-0.005574,0.021225
credit_history_years,0.002965,-0.006463,0.019765,0.001448,0.000850,0.002491,-0.013356,-0.005268,0.005044,1.000000,0.007127,0.004298,-0.001759,0.007108,0.000878,0.011831,0.130502,0.111549,0.002382,0.063103


In [23]:
correlation.to_csv(

    OUTPUT_DIR /

    "correlation_matrix.csv"

)

print("Correlation matrix saved.")

Correlation matrix saved.


In [24]:
dataset_path = (

    SYNTHETIC_DIR /

    "SAFE_MFI_Synthetic_Dataset_v1.csv"

)

dataset.to_csv(

    dataset_path,

    index=False

)

print(f"Dataset saved to:\n{dataset_path}")

Dataset saved to:
C:\Users\oluwa\data\synthetic\SAFE_MFI_Synthetic_Dataset_v1.csv


In [25]:
summary.to_csv(

    OUTPUT_DIR /

    "summary_statistics.csv"

)

print("Summary statistics exported.")

Summary statistics exported.


In [26]:
dictionary = pd.DataFrame({

    "Variable":dataset.columns,

    "Data Type":[

        str(dtype)

        for dtype in dataset.dtypes

    ],

    "Description":[

        "Generated variable"

        for _ in dataset.columns

    ]

})

dictionary.to_excel(

    SYNTHETIC_DIR /

    "data_dictionary.xlsx",

    index=False

)

dictionary.head()

,Variable,Data Type,Description
0,customer_id,object,Generated variable
1,branch_id,object,Generated variable
2,application_date,datetime64[ns],Generated variable
3,region,object,Generated variable
4,rural_urban,object,Generated variable


In [27]:
dataset_card = f"""
# SAFE-MFI Synthetic Dataset

Version: 1.0

Generated: {datetime.now().strftime('%Y-%m-%d')}

------------------------------------------------------

Dataset Name

SAFE_MFI_Synthetic_Dataset_v1

------------------------------------------------------

Purpose

Synthetic dataset developed to evaluate
the SAFE-MFI Operational AI Safety Framework
for credit risk assessment in microfinance.

------------------------------------------------------

Number of Records

{len(dataset)}

------------------------------------------------------

Number of Features

{len(dataset.columns)}

------------------------------------------------------

Target Variable

loan_status

1 = Performing Loan

0 = Default

------------------------------------------------------

Country Context

Nigeria

------------------------------------------------------

Currency

NGN

------------------------------------------------------

Generation Method

Synthetic data generated using statistical
distributions, business rules,
and probabilistic risk modelling.

------------------------------------------------------

Research Use

Academic Research

AI Safety Evaluation

Explainability

Fairness Assessment

Governance Demonstration

Operational Monitoring

------------------------------------------------------

Author

SAFE-MFI Research Project

------------------------------------------------------

License

MIT
"""

with open(

    SYNTHETIC_DIR /

    "dataset_card.md",

    "w",

    encoding="utf-8"

) as f:

    f.write(dataset_card)

print("Dataset Card Generated.")

Dataset Card Generated.


In [28]:
readme = """
# SAFE-MFI Synthetic Dataset

This folder contains the synthetic
microfinance lending dataset used
throughout the SAFE-MFI research project.

Files

SAFE_MFI_Synthetic_Dataset_v1.csv

data_dictionary.xlsx

dataset_card.md

Generation

Run

python generate_dataset.py

or execute Notebook 01.

This dataset is entirely synthetic and
contains no personally identifiable
information.

"""

with open(

    SYNTHETIC_DIR /

    "README.md",

    "w",

    encoding="utf-8"

) as f:

    f.write(readme)

print("README generated.")

README generated.


In [29]:
print("=" * 70)

print("SAFE-MFI DATASET SUCCESSFULLY GENERATED")

print("=" * 70)

print(f"Rows       : {len(dataset):,}")

print(f"Columns    : {len(dataset.columns)}")

print(f"Output File: {dataset_path}")

print("\nGenerated Files")

print("----------------")

print("SAFE_MFI_Synthetic_Dataset_v1.csv")

print("data_dictionary.xlsx")

print("dataset_card.md")

print("README.md")

print("summary_statistics.csv")

print("correlation_matrix.csv")

print("\nNotebook 01 Complete")

SAFE-MFI DATASET SUCCESSFULLY GENERATED
Rows       : 10,000
Columns    : 40
Output File: C:\Users\oluwa\data\synthetic\SAFE_MFI_Synthetic_Dataset_v1.csv

Generated Files
----------------
SAFE_MFI_Synthetic_Dataset_v1.csv
data_dictionary.xlsx
dataset_card.md
README.md
summary_statistics.csv
correlation_matrix.csv

Notebook 01 Complete
